# University Cutoff Prediction - Model Building Notebook
This notebook contains the complete pipeline for loading data, cleaning features, handling outliers by capping, and training a Random Forest Regressor.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
import joblib

# 1. Load Data
data = pd.read_csv('cutoffs.csv')
clean_data = data.dropna(subset=['figure']).copy()

# 2. Handle Outliers
cap_threshold = clean_data['figure'].quantile(0.95)
clean_data['figure_capped'] = np.minimum(clean_data['figure'], cap_threshold)

y = clean_data['figure_capped']
X = clean_data.drop(columns=['figure', 'figure_capped'])

In [ ]:
# 3. Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Pipeline Construction
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

num_prep = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_prep = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), ('onehot', OneHotEncoder(handle_unknown='ignore', max_categories=100))])

preprocessor = ColumnTransformer([('num', num_prep, num_cols), ('cat', cat_prep, cat_cols)])

best_pipeline = Pipeline([('preprocessor', preprocessor), ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))])

# 5. Fit & Export
best_pipeline.fit(X_train, y_train)
joblib.dump(best_pipeline, 'best_random_forest_model.joblib')
joblib.dump(best_pipeline.named_steps['preprocessor'], 'scaler_preprocessor.pkl')
joblib.dump(best_pipeline.named_steps['regressor'], 'model.pkl')
print('Model and scaler exported successfully!')